# PatchCore Experiments on MVTec AD

This notebook evaluates a PatchCore-inspired anomaly detection pipeline on the MVTec AD dataset.

## Goals
- build or load a memory bank from normal training samples
- compute PatchCore anomaly scores
- visualize anomaly heatmaps and overlays
- compare scoring strategies (`mean`, `max`, `topk_mean`)
- save metrics and memory banks for reproducibility

## Imports

In [ ]:
# Optional Colab setup
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader

from datasets.mvtec import MvtecAdDataset
from evaluation.metrics import compute_patchcore_scores
from models.patchcore import (
    FeatureExtractor,
    MultiScaleFeatureExtractor,
    build_memory_bank,
    random_coreset_sampling,
)
from src.config import DATA_DIR, BATCH_SIZE
from utils.normalization import preprocess_image
from visualization.heatmap import visualize_patchcore_overlay

## Device 

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Experiment Configuration

In [ ]:
CATEGORY = "capsule"  # example: capsule, bottle, screw, hazelnut
EXTRACTOR_TYPE = "multiscale"  # "layer3" or "multiscale"

BUILD_MEMORY_BANK = True
LOAD_MEMORY_BANK = False

MEMORY_BANK_PATH = ""  # fill this if LOAD_MEMORY_BANK = True

USE_RANDOM_CORESET = False
CORESET_RATIO = 0.1  # used only if USE_RANDOM_CORESET = True

SCORING_METHODS = ["mean", "max", "topk_mean"]
K_RATIO = 0.01

NUM_IMAGES_TO_DISPLAY = 5
OVERLAY_ALPHA = 0.4

RUN_NAME = (
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{CATEGORY}_patchcore_{EXTRACTOR_TYPE}"
)
print("Run name:", RUN_NAME)

## Output folders

In [ ]:
OUTPUT_DIR = Path("../outputs")
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MEMORY_DIR = OUTPUT_DIR / "memory_banks"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
MEMORY_DIR.mkdir(parents=True, exist_ok=True)

print("Figures dir:", FIGURES_DIR)
print("Metrics dir:", METRICS_DIR)
print("Memory dir:", MEMORY_DIR)

## Data Loading

In [ ]:
train_dataset = MvtecAdDataset(
    root_dir=DATA_DIR,
    category=CATEGORY,
    split="train",
    transform=preprocess_image,
)

test_dataset = MvtecAdDataset(
    root_dir=DATA_DIR,
    category=CATEGORY,
    split="test",
    transform=preprocess_image,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))
batch = next(iter(train_loader))
print("Train batch image shape:", batch["image"].shape)
print("Train batch labels shape:", batch["label"].shape)

## Feature Extractor

In [ ]:
if EXTRACTOR_TYPE == "layer3":
    extractor = FeatureExtractor().to(DEVICE)
elif EXTRACTOR_TYPE == "multiscale":
    extractor = MultiScaleFeatureExtractor().to(DEVICE)
else:
    raise ValueError("EXTRACTOR_TYPE must be 'layer3' or 'multiscale'")

extractor.eval()
print("Extractor:", extractor.__class__.__name__)

In [ ]:
x = torch.randn(1, 3, 256, 256).to(DEVICE)
with torch.no_grad():
    y = extractor(x)

print("Feature map shape:", y.shape)

## Memory Bank Construction / Loading

In [ ]:
if BUILD_MEMORY_BANK:
    memory_bank = build_memory_bank(
        feature_extractor=extractor,
        dataloader=train_loader,
        device=DEVICE,
    )
    print("Full memory bank shape:", memory_bank.shape)

    if USE_RANDOM_CORESET:
        memory_bank = random_coreset_sampling(
            memory_bank=memory_bank,
            sampling_ratio=CORESET_RATIO,
        )
        print("Random coreset memory bank shape:", memory_bank.shape)

    save_path = MEMORY_DIR / f"{RUN_NAME}.pt"
    torch.save(
        {
            "category": CATEGORY,
            "extractor_type": EXTRACTOR_TYPE,
            "use_random_coreset": USE_RANDOM_CORESET,
            "coreset_ratio": CORESET_RATIO if USE_RANDOM_CORESET else None,
            "memory_bank": memory_bank.cpu(),
        },
        save_path,
    )
    print("Memory bank saved to:", save_path)

elif LOAD_MEMORY_BANK:
    saved = torch.load(MEMORY_BANK_PATH, map_location="cpu")
    memory_bank = saved["memory_bank"]
    print("Loaded memory bank shape:", memory_bank.shape)
else:
    raise ValueError("Either BUILD_MEMORY_BANK or LOAD_MEMORY_BANK must be True.")

## Qualitative Visualization

In [ ]:
visualize_patchcore_overlay(
    feature_extractor=extractor,
    dataloader=test_loader,
    memory_bank=memory_bank,
    device=DEVICE,
    num_images=NUM_IMAGES_TO_DISPLAY,
    reduction="max",
    k_ratio=K_RATIO,
    alpha=OVERLAY_ALPHA,
)

## Quantitative Evaluation

In [ ]:
results = {}

for method in SCORING_METHODS:
    scores, labels = compute_patchcore_scores(
        feature_extractor=extractor,
        dataloader=test_loader,
        memory_bank=memory_bank,
        device=DEVICE,
        reduction=method,
        k_ratio=K_RATIO,
    )
    auc = roc_auc_score(labels, scores)
    results[method] = auc
    print(f"{CATEGORY} - {EXTRACTOR_TYPE} - {method}: {auc:.4f}")

results

In [ ]:
# ROC curve
plt.figure(figsize=(6, 6))

for method in SCORING_METHODS:
    scores, labels = compute_patchcore_scores(
        feature_extractor=extractor,
        dataloader=test_loader,
        memory_bank=memory_bank,
        device=DEVICE,
        reduction=method,
        k_ratio=K_RATIO,
    )
    fpr, tpr, _ = roc_curve(labels, scores)
    plt.plot(fpr, tpr, label=f"{method} (AUC={results[method]:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve - {CATEGORY} - {EXTRACTOR_TYPE}")
plt.legend()
plt.grid(True)
plt.show()

## Save results

In [ ]:
metrics_path = METRICS_DIR / f"{RUN_NAME}.json"

payload = {
    "category": CATEGORY,
    "extractor_type": EXTRACTOR_TYPE,
    "use_random_coreset": USE_RANDOM_CORESET,
    "coreset_ratio": CORESET_RATIO if USE_RANDOM_CORESET else None,
    "k_ratio": K_RATIO,
    "results": results,
}

with open(metrics_path, "w") as f:
    json.dump(payload, f, indent=4)

print("Metrics saved to:", metrics_path)

## Notes / Interpretation

- Heatmap quality:
- Best reduction method:
- Memory bank size:
- Comparison with previous models:
- Next action: